In [1]:
import os
import pickle
from pathlib import Path
import numpy as np
from tqdm import tqdm
import librosa
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

In [2]:
def create_mfcc_features(clean_dir, n_mfcc=13, hop_length=512):
    clean_dir = Path(clean_dir)
    X = []
    y = []
    labels = [f.name for f in clean_dir.iterdir() if f.is_dir()]

    for label in tqdm(labels):
        for audio in (clean_dir / label).iterdir():
            if audio.suffix == '.wav':
                signal, sr = librosa.load(audio, sr=None)
                n_fft = min(2048, len(signal))
                mfcc = librosa.feature.mfcc(y=signal, 
                                            sr=sr, 
                                            n_mfcc=n_mfcc, 
                                            n_fft=n_fft, 
                                            hop_length=hop_length)
                mfcc_mean = np.mean(mfcc, axis=1)
                mfcc_std = np.std(mfcc, axis=1)
                mfcc_features = np.concatenate([mfcc_mean, mfcc_std])
                X.append(mfcc_features)
                y.append(label)

    return np.array(X), np.array(y)

Load data

In [3]:
X, y = create_mfcc_features('data/IRMAS_clean')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=1)

100%|██████████| 11/11 [00:29<00:00,  2.69s/it]


Train SVM

In [4]:
# svm = SVC(kernel='linear')
# svm.fit(X_train, y_train)

In [5]:
# Load SVM
with open('models/IRMAS_svm_model.pkl', 'rb') as model_file:
    svm = pickle.load(model_file)

Evaluate SVM

In [6]:
y_pred = svm.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print('Accuracy:', accuracy)
print('\nClassification Report:\n', class_report)

output_file = 'evaluations/IRMAS_svm_evaluation.txt'

with open(output_file, 'w') as file:
    file.write(f'Accuracy: {accuracy}\n')
    file.write('\nClassification Report:\n')
    file.write(class_report)

# best .49

Accuracy: 0.4809843400447427

Classification Report:
               precision    recall  f1-score   support

         cel       0.51      0.62      0.56        78
         cla       0.54      0.42      0.47       101
         flu       0.45      0.33      0.38        90
         gac       0.45      0.49      0.47       127
         gel       0.35      0.49      0.41       152
         org       0.44      0.59      0.50       136
         pia       0.53      0.55      0.54       144
         sax       0.36      0.24      0.29       125
         tru       0.52      0.41      0.46       116
         vio       0.63      0.44      0.52       116
         voi       0.60      0.65      0.63       156

    accuracy                           0.48      1341
   macro avg       0.49      0.47      0.47      1341
weighted avg       0.49      0.48      0.48      1341



Save SVM

In [7]:
# model_filepath = 'models/IRMAS_svm_model.pkl'

# with open(model_filepath, 'wb') as model_file:
#     pickle.dump(svm, model_file)

# print(f'SVM saved to {os.path.abspath(output_file)}')